# Sleep duration → cognitive bridge analysis

This notebook tests whether **sleep duration** predicts cognitive outcomes strongly enough for the installation.

Rule: use a path only if the **linear model** has test R² > 0 and RMSE improves by at least 5% over the mean baseline.

In [1]:

from pathlib import Path
import urllib.request
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

DATA_DIR = Path('data/external_sleep_datasets')
NHANES_DIR = DATA_DIR / 'nhanes_sleep_cognitive'
NHANES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CSV = DATA_DIR / 'sleep_deprivation_dataset_detailed.csv'
RESULT_CSV = Path('sleep_duration_cognitive_bridge_results.csv')
STRICT_MIN_GAIN = 5.0
RANDOM_STATE = 42

def rmse(y, pred): return float(np.sqrt(mean_squared_error(y, pred)))
def fit_path(df, dataset, section, predictor, outcome, source_note):
    work = df[[predictor, outcome]].dropna().copy()
    work = work[np.isfinite(work[predictor]) & np.isfinite(work[outcome])]
    if len(work) < 20:
        return {'dataset':dataset,'section':section,'path':f'{predictor} → {outcome}','outcome':outcome,'predictors':predictor,'rows':len(work),'baseline_rmse':np.nan,'linear_rmse':np.nan,'linear_test_r2':np.nan,'linear_rmse_gain_pct':np.nan,'rf_rmse':np.nan,'rf_test_r2':np.nan,'rf_rmse_gain_pct':np.nan,'linear_formula':'not enough rows','decision':'do not use','source_note':source_note}
    X=work[[predictor]].astype(float); y=work[outcome].astype(float)
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.30,random_state=RANDOM_STATE)
    base_pred=np.repeat(ytr.mean(), len(yte)); base=rmse(yte, base_pred)
    lin=LinearRegression().fit(Xtr,ytr); lpred=lin.predict(Xte); lrmse=rmse(yte,lpred); lr2=float(r2_score(yte,lpred)); lgain=(base-lrmse)/base*100
    rf=RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_STATE).fit(Xtr,ytr); rpred=rf.predict(Xte); rrmse=rmse(yte,rpred); rr2=float(r2_score(yte,rpred)); rgain=(base-rrmse)/base*100
    decision='usable' if (lr2>0 and lgain>=STRICT_MIN_GAIN) else 'do not use'
    formula=f'{outcome} = {lin.intercept_:.6f} {lin.coef_[0]:+.6f} * {predictor}' if decision=='usable' else f'not usable; tested formula: {outcome} = {lin.intercept_:.6f} {lin.coef_[0]:+.6f} * {predictor}'
    return {'dataset':dataset,'section':section,'path':f'{predictor} → {outcome}','outcome':outcome,'predictors':predictor,'rows':len(work),'baseline_rmse':base,'linear_rmse':lrmse,'linear_test_r2':lr2,'linear_rmse_gain_pct':lgain,'rf_rmse':rrmse,'rf_test_r2':rr2,'rf_rmse_gain_pct':rgain,'linear_formula':formula,'decision':decision,'source_note':source_note}

def download(url, dest):
    if dest.exists() and dest.stat().st_size > 0: return True, 'already exists'
    try:
        req=urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=45) as r: dest.write_bytes(r.read())
        return True, 'downloaded'
    except Exception as e:
        return False, f'download failed: {type(e).__name__}: {e}'

results=[]
local=pd.read_csv(LOCAL_CSV)
for out in ['N_Back_Accuracy','PVT_Reaction_Time','Stroop_Task_Reaction_Time']:
    results.append(fit_path(local,'Local cognitive CSV','formal local reproduction','Sleep_Hours',out,str(LOCAL_CSV)))

files=[('2011-2012','SLQ_G','https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/SLQ_G.XPT'),('2011-2012','CFQ_G','https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/CFQ_G.XPT'),('2013-2014','SLQ_H','https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/SLQ_H.XPT'),('2013-2014','CFQ_H','https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/CFQ_H.XPT')]
status=[]
for cycle, name, url in files:
    ok,msg=download(url, NHANES_DIR/f'{name}.XPT')
    status.append({'cycle':cycle,'file':name,'url':url,'status':msg})
status_df=pd.DataFrame(status)

# If NHANES XPT files are present now or from a previous run, test likely sleep/cognitive columns automatically.
if all((NHANES_DIR/f'{n}.XPT').exists() for n in ['SLQ_G','CFQ_G','SLQ_H','CFQ_H']):
    frames=[]
    for cyc,sleep_name,cog_name in [('2011-2012','SLQ_G','CFQ_G'),('2013-2014','SLQ_H','CFQ_H')]:
        slp=pd.read_sas(NHANES_DIR/f'{sleep_name}.XPT', format='xport')
        cog=pd.read_sas(NHANES_DIR/f'{cog_name}.XPT', format='xport')
        m=slp.merge(cog,on='SEQN',how='inner'); m['cycle']=cyc; frames.append(m)
    nh=pd.concat(frames,ignore_index=True)
    sleep_candidates=[c for c in ['SLD010H','SLD012','SLQ050'] if c in nh.columns]
    outcome_candidates=[c for c in ['CFDCSR','CFDDS','CFDAST','CFDWR','CFDCRNC','CFDCRNC1','CFDCRNC2','CFDCRNC3'] if c in nh.columns]
    if sleep_candidates:
        pred=sleep_candidates[0]
        for out in outcome_candidates:
            results.append(fit_path(nh,'NHANES sleep + cognitive files','NHANES automatic test',pred,out,'Downloaded NHANES XPT files'))
else:
    for label in ['sleep duration → memory / word recall','sleep duration → processing speed / DSST','sleep duration → verbal/category fluency']:
        results.append({'dataset':'NHANES sleep + cognitive files','section':'NHANES automatic download check','path':label,'outcome':label.split('→')[-1].strip(),'predictors':'sleep duration','rows':0,'baseline_rmse':np.nan,'linear_rmse':np.nan,'linear_test_r2':np.nan,'linear_rmse_gain_pct':np.nan,'rf_rmse':np.nan,'rf_test_r2':np.nan,'rf_rmse_gain_pct':np.nan,'linear_formula':'not available; NHANES XPT download blocked or absent','decision':'do not use','source_note':'; '.join(status_df['file']+': '+status_df['status'])})

res=pd.DataFrame(results)
res.to_csv(RESULT_CSV,index=False)
print(status_df.to_string(index=False))
print(res[['dataset','path','rows','baseline_rmse','linear_rmse','linear_test_r2','linear_rmse_gain_pct','rf_rmse','rf_test_r2','rf_rmse_gain_pct','decision']].to_string(index=False))


    cycle  file                                                                   url                                                                             status
2011-2012 SLQ_G https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/SLQ_G.XPT download failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
2011-2012 CFQ_G https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/CFQ_G.XPT download failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
2013-2014 SLQ_H https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/SLQ_H.XPT download failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
2013-2014 CFQ_H https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/CFQ_H.XPT download failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
                       dataset                                     path  rows  baseline_rmse  linear_rmse  linear_test_r2  linear_rmse_gain_pct    rf_rmse 

## Decision note

Do **not** claim that sleep duration supports memory or processing speed unless the strict linear rule passes.